## Notebook onde fiz algumas visualizações dos dados para saber como eles estavam chegando em cada camada e identificar possíveis erros de tratamento

In [0]:
from pyspark.sql import functions as F, Row
from pyspark.sql.window import Window
from pyspark.sql.types import LongType, DoubleType, DecimalType
from delta.tables import DeltaTable
from datetime import datetime

In [0]:
#data de lançamento mais antiga 
display(spark.table("cineData_analytics.silver.tb_info_filmes").agg(F.min("data_lancamento"), F.max("data_lancamento")))

#padrões de data e status na origem
display(spark.table("cineData_analytics.bronze.tb_movies_info")
    .select(F.regexp_replace(F.regexp_replace("release_date", r"\d", "9"), r"[A-Za-z]", "A").alias("padrao"))
    .groupBy("padrao").count().orderBy(F.desc("count")))
display(spark.table("cineData_analytics.bronze.tb_movies_info").groupBy("status").count().orderBy(F.desc("count")))

#ruídos em orçamento e receita
display(spark.table("cineData_analytics.bronze.tb_movies_financials")
    .filter(~F.col("budget").rlike(r"^\d+$")).groupBy("budget").count().orderBy(F.desc("count")))

#popularidade que não é número simples
display(spark.table("cineData_analytics.bronze.tb_movies_metrics")
    .filter(~F.col("popularity").rlike(r"^\d+(\.\d+)?$")).select("popularity").distinct().limit(50))

#valores curtos ou longos em pessoas e produtoras
display(spark.table("cineData_analytics.silver.tb_pessoas_empresas")
    .filter(F.length("nome_entidade") > 60).select("nome_entidade", "tipo_entidade").limit(50))

min(data_lancamento),max(data_lancamento)
2016-01-01,2029-10-13


padrao,count
9999-99-99,91541
99/99/9999,10602
99-99-9999,4717
null,62
AAAAA'A AAAAAAAA AAAAAAAAA. AAAAA'A AAAAAAAA AAAAAAA.,1
AAA AAAAAA AAAA AAAAAAAAA AAAAAAA,1
AAAAA AAAAAAAA AA AAAAAAAA AAA AAAA AA AAA AAAAAAA,1
AA AAAAA AAA A AAAAA AAAAAA,1
AA AAAAAAAAAAA AAAA AAAAA AAAAA AAAAA AAAAAAA AAAAAAAAAA AAAAAAAAA,1
AAA AAAA AAAAAA AAAAAAAAA AAA AAAAAAA AAAA,1


status,count
Released,80265
RELEASED,16031
released,9107
Post Production,531
In Production,460
IN PRODUCTION,105
POST PRODUCTION,93
post production,74
null,66
in production,55


budget,count
N/A,6253
$ 10000,33
USD 10000,25
$ 5000,24
$ 800,22
$ 100000,20
$ 1000,18
$ 1000000,17
10.0K,17
USD 20000,16


popularity
"154,34"
"54,522"
"54,628"
"44,51"
"98,11"
"50,399"
"50,088"
"causing others from across the Spider-Verse to be inadvertently transported to his dimension."""
"38,002"
"52,471"


nome_entidade,tipo_entidade
Escuela Internacional De Cine Y Televisión De San Antonio De Los Baños,Produtora
Escuela Internacional De Cine Y Televisión De San Antonio De Los Baños,Produtora
Centro Regional De Formación Docente E Investigación Educativa,Produtora
The Wish Made Becomes Fatal: Everything Starts To Work Out For Valya,Produtora
Deepens Despite The Physical Distance. The Film Navigates Themes Of Identity,Produtora
Secretaria De Cultura E Economia Criativa Do Estado De São Paulo,Produtora
Secretaria De Cultura E Economia Criativa Do Estado De São Paulo,Produtora
From Victorian Farm Girl To One Of The World’s Greatest Opera Sopranos,Produtora
Zürcher Hochschule Der Künste Zhdk Departement Darstellende Künste Und Film,Produtora
And Quickly Became One Of The Most Popular Youtubers. Tamogami,Produtora


In [0]:
df_datas = spark.table("cineData_analytics.bronze.tb_movies_info").filter(F.col("release_date").rlike(r"^\d{2}[/-]\d{2}[/-]\d{4}$"))
display(
    df_datas
    .withColumn("separador", F.when(F.col("release_date").contains("/"), "/").otherwise("-"))
    .withColumn("p1", F.split("release_date", "[/-]")[0].cast("int"))
    .withColumn("p2", F.split("release_date", "[/-]")[1].cast("int"))
    .groupBy("separador")
    .agg(F.sum((F.col("p1") > 12).cast("int")).alias("primeiro_maior_12"),
         F.sum((F.col("p2") > 12).cast("int")).alias("segundo_maior_12"),
         F.count("*").alias("total"))
)

separador,primeiro_maior_12,segundo_maior_12,total
-,0,2778,4717
/,6215,0,10602


In [0]:
display(spark.table("cineData_analytics.silver.tb_financeiro_filmes").filter("id_filme = 38700"))

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
38700,90000000.00,426505244.00,374544000.00,1774944223.43,336505244.00,1400400223.43,78.90


In [0]:
display(spark.table("cineData_analytics.silver.tb_financeiro_filmes").filter("orcamento_usd < 10 OR receita_usd < 10").count())

137

In [0]:
spark.table("cineData_analytics.silver.tb_financeiro_filmes").filter(
    (F.col("orcamento_usd").isNotNull() & F.col("orcamento_brl").isNull()) |
    (F.col("receita_usd").isNotNull() & F.col("receita_brl").isNull())
).count()

display(spark.table("cineData_analytics.silver.tb_cotacao_dolar").agg(F.min("data_cotacao"), F.max("data_cotacao")))

min(data_cotacao),max(data_cotacao)
2015-01-02,2026-09-20


In [0]:
display(spark.table("cineData_analytics.silver.tb_financeiro_filmes").filter("id_filme = 38700"))

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
38700,90000000.00,426505244.00,374544000.00,1774944223.43,336505244.00,1400400223.43,78.90


In [0]:
display(spark.table("cineData_analytics.bronze.tb_movies_financials").filter("id = 38700"))

id,budget,revenue,ingestion_datetime
38700,90.0M,426505244,2026-09-20T18:07:59.358Z


In [0]:
print(spark.table("cineData_analytics.silver.tb_financeiro_filmes").filter(
    (F.col("orcamento_usd").isNotNull() & F.col("orcamento_brl").isNull()) |
    (F.col("receita_usd").isNotNull() & F.col("receita_brl").isNull())
).count())

0


In [0]:
df_baixo = (
    spark.table("cineData_analytics.silver.tb_financeiro_filmes")
    .filter("orcamento_usd < 10 OR receita_usd < 10")
    .select("id_filme", "orcamento_usd", "receita_usd")
)
df_raw = spark.table("cineData_analytics.bronze.tb_movies_financials").select(F.col("id").cast("long").alias("id_filme"), "budget", "revenue")
display(df_baixo.join(df_raw, "id_filme").limit(40))

id_filme,orcamento_usd,receita_usd,budget,revenue
619979,4.00,null,4,0
571055,1.00,null,$ 1,0
429107,null,7.00,N/A,7
383809,162300.00,3.00,162.3K,3
500735,null,1.00,0,1
488621,null,3.00,0,3
407620,5.00,null,5,Unknown
467028,null,2.00,0,2
476996,2.00,null,2,0
681711,1.00,null,1,0


In [0]:
df_info = spark.table("cineData_analytics.silver.tb_info_filmes").select("id_filme")
df_fin = spark.table("cineData_analytics.silver.tb_financeiro_filmes")
df_met = spark.table("cineData_analytics.silver.tb_metricas_engajamento")

print("filmes em info:", df_info.count())
print("financeiro sem info:", df_fin.join(df_info, "id_filme", "left_anti").count())
print("metricas sem info:", df_met.join(df_info, "id_filme", "left_anti").count())
print("info sem financeiro:", df_info.join(df_fin.select("id_filme"), "id_filme", "left_anti").count())

filmes em info: 97879
financeiro sem info: 1264
metricas sem info: 1264
info sem financeiro: 137


In [0]:
df_orfaos = df_fin.join(df_info, "id_filme", "left_anti")
display(df_orfaos.agg(F.count("*").alias("filmes"), F.sum("receita_brl").alias("receita_brl_orfaos")))
display(df_fin.agg(F.sum("receita_brl").alias("receita_brl_total")))

filmes,receita_brl_orfaos
1264,1413931956.01


receita_brl_total
664274561387.69


In [0]:
catalog = "cineData_analytics"

for t in ["tb_movies_info", "tb_movies_financials", "tb_movies_metrics", "tb_credits_and_tags", "tb_movies_reviews", "tb_cotacao_dolar"]:
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.bronze.{t}")

for t in ["tb_info_filmes", "tb_cotacao_dolar", "tb_financeiro_filmes", "tb_metricas_engajamento", "tb_avaliacoes_usuarios", "tb_generos", "tb_pessoas_empresas", "dq_log"]:
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.silver.{t}")

for t in ["dim_movies", "dim_genres", "dim_people", "dim_companies", "dim_reviews", "bridge_movie_genre", "bridge_movie_person", "bridge_movie_company", "fact_movies_performance", "gold_genai_movies_context"]:
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.gold.{t}")